# A person has a level, and so does a movie

The global mean treats every rating as exchangeable. The next two summaries do not.

User 405 has the longest row: 737 ratings. She is 22, female, occupation healthcare, zip code 10019. User 1's 272 ratings average $491/136$. Those two people can sit on different parts of the 1-to-5 scale even before you know which movie is which.

*Star Wars (1977)* is movie 50, the longest column: 583 ratings. Their sum is 2541.

$$
\dfrac{2541}{583} = \dfrac{231}{53}.
$$

That is $4 + 19/53$, a little above 4. The next four columns by length are *Contact (1997)* with 509 ratings, *Fargo (1996)* with 508, *Return of the Jedi (1983)* with 507, and *Liar Liar (1997)* with 485.

A user-mean prediction says: for a movie this person has not rated, guess the mean of the movies they have rated. A movie-mean prediction says: guess the mean of the people who have rated this movie. Each ignores the other side of the pair. The next lesson puts the two means in one formula.


In [2]:
# Locate the MovieLens tables beside this notebook, or under the course root.
from pathlib import Path

def data_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "16-Movie Recommendation" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "u.info").exists():
            return direct
    raise FileNotFoundError(name)

def read_lines(name):
    return data_path(name).read_text(encoding="latin-1").splitlines()

def read_ratings(name):
    rows = []
    for line in read_lines(name):
        if not line.strip():
            continue
        user, item, rating, stamp = line.split("\t")
        rows.append((int(user), int(item), int(rating), int(stamp)))
    return rows

ratings = read_ratings("u.data")
print(len(ratings), ratings[0])


100000 (196, 242, 3, 881250949)


In [3]:
# Longest user, user 1's mean, and the five longest movies.
from fractions import Fraction
from collections import Counter, defaultdict
users = {int(row[0]): row for row in (line.split("|") for line in read_lines("u.user"))}
movies = {int(row[0]): row[1] for row in (line.split("|") for line in read_lines("u.item"))}
by_user = Counter()
by_movie = defaultdict(list)
for user, item, rating, _ in ratings:
    by_user[user] += 1
    by_movie[item].append(rating)
busiest, longest = by_user.most_common(1)[0]
print(busiest, longest, users[405])
assert (busiest, longest) == (405, 737)
assert users[405] == ["405", "22", "F", "healthcare", "10019"]
user_one = [rating for user, _, rating, _ in ratings if user == 1]
assert Fraction(sum(user_one), len(user_one)) == Fraction(491, 136)
star = by_movie[50]
print(movies[50], len(star), sum(star), Fraction(sum(star), len(star)))
assert movies[50] == "Star Wars (1977)"
assert len(star) == 583 and sum(star) == 2541
assert Fraction(2541, 583) == Fraction(231, 53)
longest_movies = sorted(by_movie, key=lambda item: len(by_movie[item]), reverse=True)[:5]
print([(item, movies[item], len(by_movie[item])) for item in longest_movies])
assert longest_movies == [50, 258, 100, 181, 294]
assert [movies[item] for item in longest_movies] == [
    "Star Wars (1977)",
    "Contact (1997)",
    "Fargo (1996)",
    "Return of the Jedi (1983)",
    "Liar Liar (1997)",
]


405 737 ['405', '22', 'F', 'healthcare', '10019']
Star Wars (1977) 583 2541 231/53
[(50, 'Star Wars (1977)', 583), (258, 'Contact (1997)', 509), (100, 'Fargo (1996)', 508), (181, 'Return of the Jedi (1983)', 507), (294, 'Liar Liar (1997)', 485)]


## Pitfall

*Star Wars* averaging $231/53$ does not mean user 405 would give it that score. Her row has its own level. A movie mean throws her level away, and a user mean throws the movie away.

## Summary

User 405 rated 737 movies. User 1's mean is $491/136$. *Star Wars* was rated 583 times and averages $231/53$.
